*   This notebook implements softmax regression directly with tensors so every part of classification training is visible before framework shortcuts.

In [1]:
import math
import random
import numpy as np
import torch

# 4.4.1 The Softmax

## 1. Intuition

*   Softmax converts one row of class logits into probabilities.

*   It exponentiates each score and divides by the sum of exponentiated scores in that row.

## 2. Why this exists

*   A classifier needs comparable class probabilities for cross-entropy loss and interpretation.

## 3. Examples

*   Define softmax for a batch of logits.

In [2]:
def softmax(logits):
  shifted = logits - logits.max(dim=1, keepdim=True).values # Use logits to minus the largest in logit (2), or shifted = [1-2, 2-2, 0-2] = [-1, 0, -2]. The largest value is now 0.
  exp_logits = torch.exp(shifted) # Convert them into exponents to prepare for softmax [e^-1, e^0, e^-2]
  return exp_logits / exp_logits.sum(dim=1, keepdim=True) # Use softmax by dividing each element of exp_logits over its sum

logits = torch.tensor([[1.0, 2.0, 0.0]])
softmax(logits)

tensor([[0.2447, 0.6652, 0.0900]])

## 4. Step-by-step breakdown

*   `logits.max(..., keepdim=True)` finds the largest score in each row.

*   Subtracting the row maximum improves numerical stability without changing softmax probabilities.

*   `torch.exp` makes scores positive.

*   Dividing by the row sum makes each row add to 1.

## 5. Connection to ML systems

*   Stable softmax is used inside classification losses.
*   PyTorch's built-in cross-entropy combines this idea with log loss efficiently.

## 6. Common confusion points

- Softmax is applied across classes, not across examples.
- Subtracting the maximum is a stability trick, not a model change.
- Softmax outputs probabilities but does not choose a class by itself.
- Very large logits can cause numerical problems without stabilization.

# 4.4.2 The Model

## 1. Intuition

*   The softmax-regression model first computes linear class scores, then converts them to probabilities.

*   The learnable parameters are a weight matrix and a bias vector.

## 2. Why this exists

*   The model must produce one class score per example per class.
*   For images, flattened pixels become input features.

## 3. Examples

*   Define a softmax-regression model from scratch.

In [4]:
num_inputs = 4
num_outputs = 3

W = torch.randn(num_inputs, num_outputs) * 0.01
W.requires_grad_()
b = torch.zeros(num_outputs, requires_grad=True)

def net(X):
  X = X.reshape(X.shape[0], -1) # Keep the batch (1st) dimension and flatten all remaining dimensions into features.
  return softmax(X @ W + b)

*   Run the model on two tiny images.

In [5]:
X = torch.randn(2, 1, 2, 2)
y_hat = net(X) # (2, 1*2*2) or (2, 4) @ (4, 3) + (3,) = (2, 3)
y_hat.shape, y_hat.sum(dim=1) # y_hat.shape = (2, 3); y_hat.sum(dim=1) = tensor([1., 1.]) because the class probabilities for each sample sum to 1.

(torch.Size([2, 3]), tensor([1., 1.], grad_fn=<SumBackward1>))

## 4. Step-by-step breakdown

* `num_inputs` is the flattened feature count.

* `num_outputs` is the number of classes.

* `W` maps input features to class scores.

* `b` adds one bias per class.

* `X.reshape(X.shape[0], -1)` keeps the batch size and flattens each example.

* `softmax(X @ W + b)` returns probabilities.

## 5. Connection to ML systems

*   This is the manual version of a linear classifier followed by softmax.

## 6. Common confusion points

- The weight matrix shape is `(inputs, classes)` in this formulation.
- The bias vector has one value per class.
- Flattening must preserve the batch dimension.
- `requires_grad_()` turns gradient tracking on for `W` after its small random initialization.

# 4.4.3 The Cross-Entropy Loss

## 1. Intuition

*   Cross-entropy selects the predicted probability assigned to the correct class and takes negative log.

*   For a batch, the loss is usually averaged or summed across examples.

## 2. Why this exists

*   The loss should become small when the model assigns high probability to the true class.

## 3. Examples

*   Define cross-entropy from predicted probabilities.

In [6]:
def cross_entropy(y_hat, y):
  row = torch.arange(y_hat.shape[0]) # In this example, y_hat.shape = (2,3) so row = torch([0, 1]) from torch.arange(2)
  return -torch.log(y_hat[row, y]) # y_hat advance index to y's [1, 0], which are [0.8, 0.7] respectively, then performs -log([0.8, 0.7])

y_hat = torch.tensor([[0.1, 0.8, 0.1], [0.7, 0.2, 0.1]])
y = torch.tensor([1, 0])
cross_entropy(y_hat, y)

tensor([0.2231, 0.3567])

## 4. Step-by-step breakdown

*   `torch.arange(y_hat.shape[0])` creates row indices for the batch.

*   `y` stores the correct class index for each row.

*   `y_hat[row, y]` selects the correct-class probability for each example.

*   `-torch.log(...)` converts those probabilities into losses.

## 5. Connection to ML systems

*   This manual loss explains what `CrossEntropyLoss` is trying to optimize, even though PyTorch's implementation expects logits and is more stable.

*   PyTorch's `CrossEntropyLoss` takes the raw model outputs (logits) directly and combines the softmax and loss calculation into a more numerically stable operation.

## 6. Common confusion points

- The manual version expects probabilities, not logits.
> * Logits are the raw outputs of the model before they have been passed through softmax.
> * PyTorch's `CrossEntropyLoss` already combines `softmax` and `crossentropy` together.
> * `softmax` smooth over classes into probabilities, and `crossentropy` uses the negative log of the probability assigned to the correct class to calculate the loss.
- The label tensor should contain integer class IDs.
- Low correct-class probability gives high loss.
- Logarithm of zero is invalid, so stable implementations avoid exact zero probabilities.

# 4.4.4 Training

## 1. Intuition

* Training softmax regression repeats the same pattern as regression: read a batch, predict, compute loss, call backward, update parameters.

* The difference is the prediction and loss are classification-specific.

## 2. Why this exists

*   The from-scratch loop proves that classification training is still ordinary tensor computation plus gradients.

## 3. Examples

*   A tiny training step on synthetic image-like data.

In [7]:
X = torch.randn(4, 1, 2, 2)
y = torch.tensor([0, 1, 2, 1])
y_hat = net(X) # Forward pass: flattens X into 2 dimensions (2nd with all features), then X @ W + b, then softmax to convert logits into class probabilities

loss = cross_entropy(y_hat, y).mean() # Calculate loss via crossentropy (-log(correct probability))
loss.backward() # Computes gradients for weight and bias via chain-rule

with torch.no_grad(): # Performing gradient descent (off of the gradients distributed via loss.backward() or W.grad and b.grad), or optimizer.step()
  W -= 0.1 * W.grad # new param = old param − learning rate × gradient
  b -= 0.1 * b.grad
  W.grad.zero_(); b.grad.zero_()  # Reset gradients before the next training step
loss

tensor(1.0860, grad_fn=<MeanBackward0>)

## 4. Step-by-step breakdown

* The batch contains four tiny image-like examples.

* `net(X)` returns class probabilities.

* `cross_entropy(...).mean()` creates one scalar loss.

* `loss.backward()` computes gradients.

* The `torch.no_grad()` block updates parameters without tracking the update as part of the graph.

* Gradients are cleared after the update.

## 5. Connection to ML systems

*   This is the same control flow used later with PyTorch optimizers, only written by hand.

## 6. Common confusion points

- The loss must be scalar before the simplest `.backward()` call.
- Parameter updates should not be tracked by autograd.
- Gradients must be cleared before the next step.
- This manual loop uses probabilities, while PyTorch's built-in loss usually uses logits.

# 4.4.5 Prediction

## 1. Intuition

* Prediction means choosing the class with the highest score or probability.

* For softmax outputs, the largest probability corresponds to the predicted class.

## 2. Why this exists

*   Training optimizes probability quality, but many applications need a final class decision.

## 3. Examples

*   Predict class IDs from probabilities.

In [8]:
probs = net(torch.randn(3, 1, 2, 2)) # Flattens to (3, 1*2*2) or (3, 4), then (3, 4) @ W + b, or (3, 4) @ (4, 3) + (3, ) = (3, 3), then apply softmax to convert logits into class probabilities
preds = torch.argmax(probs, dim=1) # # Select the class index with the highest probability for each sample
preds # Highest probability in index[0, 1, 2] of each sample

tensor([0, 1, 2])

*   Map predicted IDs to class names.

In [9]:
class_names = ["top", "trouser", "shoe"]
names = [class_names[int(i)] for i in preds]
names

['top', 'trouser', 'shoe']

## 4. Step-by-step breakdown

* `torch.argmax(probs, dim=1)` chooses the largest class probability in each row.

* The result is a tensor of class IDs.

* The list comprehension maps each ID to a readable class name.

## 5. Connection to ML systems

* Prediction (inference) code is usually used during validation, testing, and deployment. It should not update parameters.

## 6. Common confusion points

- Prediction (inference) is different from training.
- `argmax` removes probability confidence information.
- Class-name lists must match the class-ID order.
- Use `dim=1` when columns are classes.

# 4.4.6 Summary

## 1. Intuition

*   From-scratch softmax regression contains linear logits, softmax probabilities, cross-entropy loss, and gradient updates.

## 2. Why this exists

*   Seeing the full implementation makes concise PyTorch classifier code easier to audit.

## 3. Examples

*   The from-scratch classification pipeline.

In [ ]:
pipeline = [
    "flatten inputs",
    "compute logits",
    "softmax probabilities",
    "cross-entropy loss",
    "gradient update",
]

## 4. Step-by-step breakdown

* The pipeline lists the execution order.

* Each step produces values used by the next step.

* Training repeats this pipeline over batches.

## 5. Connection to ML systems

*   More complex classifiers keep the same output-and-loss pattern even when the model body changes.

## 6. Common confusion points

- Softmax regression is linear before softmax.
- Cross-entropy is the training signal.
- Accuracy is an evaluation / inference metric, not the differentiable loss used for training.
- Built-in implementations are more stable than naive manual formulas.

# 4.4.7 Exercises

## 1. Intuition

*   These exercises check the pieces of a manual classifier.

## 2. Why this exists

*   Debugging classification is easier when logits, probabilities, labels, and predictions are inspected separately.

## 3. Examples

*   Exercise 1: compute probabilities for two logit rows.

In [10]:
logits = torch.tensor([[1.0, 0.0, 2.0], [0.5, 0.5, 0.5]])
probs = softmax(logits)
probs.sum(dim=1) # [1, 1]

tensor([1., 1.])

*   Exercise 2: compute predicted classes.
* If the probability is equally high for all classes, it means the model is equally confident in every class. It is essentially saying: I don't know which class this is.
* For a classification problem, that is not a good prediction (unless the task is genuinely impossible).

In [11]:
preds = torch.argmax(probs, dim=1) # [2, 0], the second row has equal scores, so argmax returns the first maximum index.
preds

tensor([2, 0])

## 4. Step-by-step breakdown

* Exercise 1 checks row-wise normalization.

* Exercise 2 checks class selection.

* The second row has equal scores, so `argmax` returns the first maximum index.

## 5. Connection to ML systems

*   These same checks are useful when validating a real classifier's outputs.

## 6. Common confusion points

- Equal logits create equal probabilities.
- `argmax` tie-breaking can matter.
- Probabilities should be checked along the class dimension.
- Manual softmax is for learning; built-in loss is preferred in real training.